# KAT-VAD v7 — **Step 0**: Latent Surprise Predictor (D1 → stage 1 → D2–D5)

Rules: `core/docs/v7/PREREG_V7.md` **§1** (V2-S, read: NO-GO) and **§1.1** (Amendment 1: CLIP, same bars).
Pick the stream in the setup cell: `STREAM = 'clip'` (§1.1) or `'v2s'` (§1). Each stream writes its own folder
(`step0_clip/` vs `step0/`). Plan `.project/plans/katvad-v7-lsp.md` §4.
No detector is trained here; nothing is extracted (every input is an existing cache).

| step | what | GPU? |
|---|---|---|
| 1 | setup, deps, inputs located + staged to VM disk | — |
| 2 | **D1** `cos(u_t, u_{t+k})` on the 404 training negatives (before any training) | no |
| 3 | **stage 1**: LSP for K = 16 and K = 32, best epoch on the 45 held-out negatives, K chosen once | **yes** |
| 4 | **D2–D5** + decision → **paste `REPORTS/step0_readout.md` (and `d1.md`) to Claude** | yes (light) |

Every step skips if its output is already on Drive. **Drive:** code `Thesis-V7/kat-vad` (upload branch `v7`); writes
only `Thesis-V7/outputs/v7/v7_lsp/step0/`; reads `Thesis/` (DoTA labels, DoTA-CAP V2-S cache) and `Thesis-V2/` (Nexar V2-S cache, census)
read-only.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data (DoTA metadata, annotations)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: V2-S caches, Nexar census
DRIVE_V7 = '/content/drive/MyDrive/Thesis-V7'     # code + everything v7 writes
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_V2}/cache'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V7}/outputs'

REPO = Path(f'{DRIVE_V7}/kat-vad')
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v7 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V7_HORIZON'), 'this upload predates v7 -- re-upload branch v7'
assert (REPO / 'core/tools/v7_lsp.py').is_file(), 'no core/tools/v7_lsp.py in this upload'
assert (REPO / 'core/docs/v7/PREREG_V7.md').is_file(), 'no PREREG_V7.md in this upload'
HOLDOUT = constants.V7_SPLITS_DIR / constants.V7_HOLDOUT_FILENAME
assert HOLDOUT.is_file(), f'missing {HOLDOUT}'

STREAM = 'clip'      # §1.1 Amendment 1 (CLIP frame vectors); 'v2s' = §1 (already read: NO-GO)
assert STREAM in constants.V7_STREAMS
assert '## §1.1 Amendment 1' in (REPO / 'core/docs/v7/PREREG_V7.md').read_text(), 'upload predates §1.1'
SUB = constants.V7_STREAM_OUT[STREAM]
OUT_DRIVE = Path(f'{DRIVE_V7}/outputs') / constants.V7_OUTPUT_DIRNAME / constants.V7_LSP_DIRNAME / SUB
OUT = Path(f'/content/v7_{SUB}')                  # VM-local; published to OUT_DRIVE after every step
print('code', REPO, '| outputs', OUT_DRIVE)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
python -c "import sklearn, einops, faiss; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 3 needs one'

In [ ]:
import json
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd):
    '''Stream a child's output here; raise with its tail on failure.'''
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}')


def publish():
    '''Mirror VM-local outputs to Drive (file by file, tmp + rename).'''
    for src in OUT.rglob('*'):
        if src.is_file():
            dst = OUT_DRIVE / src.relative_to(OUT)
            dst.parent.mkdir(parents=True, exist_ok=True)
            tmp = dst.with_name(dst.name + '.partial')
            shutil.copy2(src, tmp)
            os.replace(tmp, dst)
    print('published ->', OUT_DRIVE)


if OUT_DRIVE.is_dir() and not OUT.exists():       # resume: bring earlier outputs back to the VM
    shutil.copytree(OUT_DRIVE, OUT)
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Inputs: census + the two V2-S caches (L0-1). Stage only the ids Step 0 reads.
ENC = constants.V7_ENCODER
CACHE_ROOTS = [Path(DRIVE_V2) / 'cache', Path(DRIVE) / 'cache']   # Nexar: Thesis-V2; DoTA-CAP: Thesis (v2 notebook)


def find_cache(name, kind):
    sub = Path('clip') if kind == 'clip' else Path('video') / ENC
    hits = [r / sub / name for r in CACHE_ROOTS if (r / sub / name).is_dir()]
    assert hits, f'missing cache {name} under {[str(r) for r in CACHE_ROOTS]} -- tell Claude where it is'
    print(f'  {name}: {hits[0]}')
    return hits[0]


if STREAM == constants.V7_STREAM_CLIP:
    NEXAR_SRC = find_cache(constants.NEXAR_CLIP_S1_DATASET, 'clip')
    DOTA_SRC  = find_cache(f'{constants.DOTA_DATASET}_s1_ncc', 'clip')
else:
    NEXAR_SRC = find_cache(f'{constants.NEXAR_VIDEO_S1_DATASET}_squash', 'video')
    DOTA_SRC  = find_cache(f'{constants.DOTA_CAP_DATASET}_s1_squash', 'video')
CENSUS_CANDIDATES = [Path(DRIVE_V2) / 'outputs/v2/REPORTS/nexar_n0/census.json',
                     Path(DRIVE_V2) / 'outputs/REPORTS/nexar_n0/census.json',
                     Path(DRIVE_V2) / 'REPORTS/nexar_n0/census.json']
CENSUS = next((p for p in CENSUS_CANDIDATES if p.is_file()), None)
assert CENSUS is not None, f'no Nexar census.json in {CENSUS_CANDIDATES} -- tell Claude where it is'

from core.data.v2_splits import load_split  # noqa: E402

census = json.loads(CENSUS.read_text())['videos']
train = load_split(constants.V2_SPLIT_NEXAR_TRAIN)
val = load_split(constants.V2_SPLIT_NEXAR_VAL)
dev = load_split(constants.V2_SPLIT_DOTA_CAP_DEV)
nexar_ids = sorted([v for v in train if census[v]['label'] == 0] + val)
print(f'census {CENSUS}; Nexar ids to stage {len(nexar_ids)} (train negatives + nexar_val), DoTA-CAP-dev {len(dev)}')

NEXAR_VM = Path('/content/cache') / NEXAR_SRC.name
DOTA_VM = Path('/content/cache') / DOTA_SRC.name


def stage(src, dst, ids):
    dst.mkdir(parents=True, exist_ok=True)
    todo = [v for v in ids if not (dst / f'{v}.npy').exists() and (src / f'{v}.npy').exists()]
    t0 = time.time()
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda v: shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy'), todo))
    absent = [v for v in ids if not (dst / f'{v}.npy').exists()]
    print(f'  {src.name}: staged {len(todo)} in {time.time() - t0:.0f}s; absent on Drive {len(absent)} {absent[:5]}')
    return absent


assert not stage(NEXAR_SRC, NEXAR_VM, nexar_ids), 'Nexar caches missing -- paste this to Claude'
DOTA_ABSENT = stage(DOTA_SRC, DOTA_VM, dev)        # absent DoTA clips are listed in the read-out
COMMON = ['--stream', STREAM, '--census', CENSUS, '--video-dir', NEXAR_VM, '--out-dir', OUT, '--device', 'auto']

## Step 2 — D1 (before any training)

In [ ]:
if not (OUT / 'REPORTS/d1.json').exists():
    run([sys.executable, '-m', 'core.tools.v7_lsp', 'd1', *COMMON])
    publish()
print((OUT / 'REPORTS/d1.md').read_text()[:600])

## Step 3 — stage 1 (GPU): K = 16 and K = 32, 50 epochs each

In [ ]:
if not (OUT / 'models/lsp_selected.json').exists():
    import torch
    assert torch.cuda.is_available(), 'stage 1 needs a GPU runtime'
    run([sys.executable, '-m', 'core.tools.v7_lsp', 'train', *COMMON])
    publish()
print(json.dumps(json.loads((OUT / 'models/lsp_selected.json').read_text()), indent=1))

## Step 4 — D2–D5 + decision → paste `step0_readout.md` to Claude

In [ ]:
if not (OUT / 'REPORTS/step0_readout.json').exists():
    run([sys.executable, '-m', 'core.tools.v7_lsp', 'diag', *COMMON, '--dota-video-dir', DOTA_VM])
    publish()
print((OUT / 'REPORTS/step0_readout.md').read_text())